In [1]:
from lib.tokenizers.raw_tokenizers import RegexBPE, RegexDaskBPE
import pickle

In [2]:
with open('vocab.pickle', 'rb') as file:
    vocab = pickle.load(file)

with open('merge_dict.pickle', 'rb') as file:
    merge_dict = pickle.load(file)

vocab

{0: b'\x00',
 1: b'\x01',
 2: b'\x02',
 3: b'\x03',
 4: b'\x04',
 5: b'\x05',
 6: b'\x06',
 7: b'\x07',
 8: b'\x08',
 9: b'\t',
 10: b'\n',
 11: b'\x0b',
 12: b'\x0c',
 13: b'\r',
 14: b'\x0e',
 15: b'\x0f',
 16: b'\x10',
 17: b'\x11',
 18: b'\x12',
 19: b'\x13',
 20: b'\x14',
 21: b'\x15',
 22: b'\x16',
 23: b'\x17',
 24: b'\x18',
 25: b'\x19',
 26: b'\x1a',
 27: b'\x1b',
 28: b'\x1c',
 29: b'\x1d',
 30: b'\x1e',
 31: b'\x1f',
 32: b' ',
 33: b'!',
 34: b'"',
 35: b'#',
 36: b'$',
 37: b'%',
 38: b'&',
 39: b"'",
 40: b'(',
 41: b')',
 42: b'*',
 43: b'+',
 44: b',',
 45: b'-',
 46: b'.',
 47: b'/',
 48: b'0',
 49: b'1',
 50: b'2',
 51: b'3',
 52: b'4',
 53: b'5',
 54: b'6',
 55: b'7',
 56: b'8',
 57: b'9',
 58: b':',
 59: b';',
 60: b'<',
 61: b'=',
 62: b'>',
 63: b'?',
 64: b'@',
 65: b'A',
 66: b'B',
 67: b'C',
 68: b'D',
 69: b'E',
 70: b'F',
 71: b'G',
 72: b'H',
 73: b'I',
 74: b'J',
 75: b'K',
 76: b'L',
 77: b'M',
 78: b'N',
 79: b'O',
 80: b'P',
 81: b'Q',
 82: b'R',
 83: b'

In [3]:
rbpe = RegexBPE()
rbpe.merge_dict = merge_dict
rbpe.vocab = vocab

In [17]:
%time rbpe.encode(" ".join(["yo yo yo what up it's ya boi sam"]*100000), pbar=True)

  1%|          | 9/820 [00:04<07:18,  1.85it/s]


KeyboardInterrupt: 

In [11]:
dask_rbpe = RegexDaskBPE()
dask_rbpe.merge_dict = merge_dict
dask_rbpe.vocab = vocab

In [16]:
from dask.distributed import Client
# client = Client(n_workers=8, threads_per_worker=1)
%time dask_rbpe.encode(" ".join(["yo yo yo what up it's ya boi sam"]*100000), pbar =True)

100%|██████████| 820/820 [00:02<00:00, 323.89it/s]


KeyboardInterrupt: 

In [21]:
from dask import bag
chunks = dask_rbpe.chunk_input(" ".join(["yo yo yo what up it's ya boi sam"]*1000))
chunk_bag = bag.from_sequence(chunks, npartitions=32)
encoded_chunks = chunk_bag.map(dask_rbpe.encode_chunk).persist()

In [19]:
dask_rbpe.save_model("../data/tokenizers/dask_tokenizer.pkl")

In [23]:
new_rbpe = pickle.load(open("../data/tokenizers/raw_python_tokenizer.pkl", "rb"))

In [30]:
new_rbpe.encode(" boy.")

[501, 46]